# Algorithmic Option Pricing: Stochastic Calculus & Fourier Transforms
**Stamatics, IIT Kanpur**  
**Mentors:** Utkarsh Sawarn · Aryan Singh · Harsh Vardhan

---

This notebook implements a full European Call Option pricing engine:
1. Black-Scholes-Merton (BSM) — analytical closed-form
2. Variance Gamma (VG) — via Carr-Madan FFT methodology
3. Volatility Smile extraction and visualisation
4. Parameter sensitivity analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import sys
sys.path.insert(0, '..')

from bsm import bsm_call_price, bsm_call_vector, implied_volatility
from variance_gamma import vg_char_func
from fft_pricer import carr_madan_fft

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Imports successful')

## 1. Black-Scholes-Merton Model

Derived from Itô's Lemma and no-arbitrage conditions:

$$C = S_0 N(d_1) - K e^{-rT} N(d_2)$$

$$d_1 = \frac{\ln(S_0/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
# Parameters
S0    = 100.0   # Current stock price
K     = 100.0   # Strike price (ATM)
r     = 0.05    # Risk-free rate
T     = 1.0     # Time to maturity (1 year)
sigma = 0.20    # Volatility (20%)

# Single call price
price = bsm_call_price(S0, K, r, T, sigma)
print(f'BSM ATM Call Price (S0=K={S0}): {price:.4f}')

# Price across a range of strikes
strikes = np.linspace(60, 160, 200)
bsm_prices = bsm_call_vector(S0, strikes, r, T, sigma)

plt.plot(strikes, bsm_prices, color='tomato', lw=2)
plt.xlabel('Strike Price (K)')
plt.ylabel('Call Price')
plt.title('BSM European Call Price vs Strike')
plt.tight_layout()

## 2. Variance Gamma Characteristic Function

The VG process adds **skewness** ($\theta$) and **excess kurtosis** ($\nu$) to the BSM framework via Gamma time-change:

$$\phi_T^{VG}(u) = e^{i u \omega T} \left(1 - i u \theta \nu + \frac{\sigma^2 \nu u^2}{2}\right)^{-T/\nu}$$

where $\omega = \frac{1}{\nu} \ln\left(1 - \theta\nu - \frac{\sigma^2\nu}{2}\right)$ is the martingale correction.

In [ ]:
# VG Parameters
nu    = 0.10    # Kurtosis (variance rate)
theta = -0.14   # Skewness (negative = left-skewed, realistic for equities)

# Plot characteristic function magnitude
u = np.linspace(0, 30, 500)
phi = vg_char_func(u, T, sigma, nu, theta, r)

plt.plot(u, np.abs(phi), color='steelblue', lw=2)
plt.xlabel('Frequency u')
plt.ylabel('|phi(u)|')
plt.title('Variance Gamma Characteristic Function Magnitude')
plt.tight_layout()

## 3. Carr-Madan FFT Pricing

Prices the entire strike chain simultaneously in **O(N log N)** via:

$$C_T(k) \approx \frac{e^{-\alpha k}}{\pi} \text{Re}\left[\text{FFT}\left(e^{ibv_j} \psi_T(v_j) \frac{\eta}{3} w_j\right)\right]$$

where $\psi_T(v) = \frac{e^{-rT} \phi_T(v-(\alpha+1)i)}{\alpha^2+\alpha-v^2+i(2\alpha+1)v}$

In [ ]:
# Run FFT pricer
fft_strikes, vg_prices = carr_madan_fft(S0, r, T, sigma, nu, theta)

# Get BSM prices for same strikes
bsm_fft = bsm_call_vector(S0, fft_strikes, r, T, sigma)

# Filter to reasonable strike range
mask = (fft_strikes > 60) & (fft_strikes < 160)

plt.plot(fft_strikes[mask], vg_prices[mask], label='Variance Gamma (FFT)', color='steelblue', lw=2)
plt.plot(fft_strikes[mask], bsm_fft[mask], label='Black-Scholes-Merton', color='tomato', lw=2, ls='--')
plt.xlabel('Strike Price (K)')
plt.ylabel('Call Option Price')
plt.title('European Call: VG (FFT) vs BSM')
plt.legend()
plt.tight_layout()

## 4. Volatility Smile

Back out **implied volatility** from VG prices using BSM as the inversion model.  
The resulting "smile" shows where BSM systematically under/overprices relative to VG.

In [ ]:
strikes_f = fft_strikes[mask]
vg_f = vg_prices[mask]

ivols = np.array([
    implied_volatility(c, S0, k, r, T)
    for c, k in zip(vg_f, strikes_f)
])

valid = ~np.isnan(ivols)
moneyness = strikes_f[valid] / S0

plt.plot(moneyness, ivols[valid] * 100, color='steelblue', lw=2.5, label='Implied Vol (VG)')
plt.axhline(y=sigma * 100, color='tomato', ls='--', lw=1.8, label=f'BSM Flat Vol ({sigma*100:.0f}%)')
plt.axvline(x=1.0, color='grey', ls=':', lw=1.2, label='ATM')
plt.xlabel('Moneyness (K/S0)')
plt.ylabel('Implied Volatility (%)')
plt.title('Volatility Smile: VG Model vs BSM Flat Volatility')
plt.legend()
plt.tight_layout()

## 5. Parameter Sensitivity

How do the VG parameters shape the volatility smile?
- **ν (nu)**: Controls kurtosis — higher ν → fatter tails → more pronounced smile
- **θ (theta)**: Controls skewness — negative θ tilts the smile (left skew, realistic for equities)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kurtosis effect
ax = axes[0]
for nu_val in [0.05, 0.10, 0.20, 0.40]:
    s, vg = carr_madan_fft(S0, r, T, sigma, nu_val, -0.1)
    m = (s > 70) & (s < 140)
    iv = np.array([implied_volatility(c, S0, k, r, T) for c, k in zip(vg[m], s[m])])
    valid = ~np.isnan(iv)
    ax.plot(s[m][valid] / S0, iv[valid] * 100, lw=2, label=f'ν = {nu_val}')
ax.axhline(y=sigma * 100, color='black', ls='--', lw=1.2, label='BSM flat')
ax.set_xlabel('Moneyness'); ax.set_ylabel('Implied Vol (%)')
ax.set_title('Effect of Kurtosis (ν)'); ax.legend(fontsize=8)

# Skewness effect
ax = axes[1]
for th_val in [0.05, 0.0, -0.10, -0.20]:
    s, vg = carr_madan_fft(S0, r, T, sigma, 0.1, th_val)
    m = (s > 70) & (s < 140)
    iv = np.array([implied_volatility(c, S0, k, r, T) for c, k in zip(vg[m], s[m])])
    valid = ~np.isnan(iv)
    ax.plot(s[m][valid] / S0, iv[valid] * 100, lw=2, label=f'θ = {th_val}')
ax.axhline(y=sigma * 100, color='black', ls='--', lw=1.2, label='BSM flat')
ax.set_xlabel('Moneyness'); ax.set_ylabel('Implied Vol (%)')
ax.set_title('Effect of Skewness (θ)'); ax.legend(fontsize=8)

plt.tight_layout()

## 6. Summary

| | BSM | Variance Gamma (FFT) |
|---|---|---|
| **Volatility** | Constant | Stochastic via Gamma time-change |
| **Distribution** | Log-normal | Skewed + fat-tailed |
| **Vol Smile** | Cannot explain | Captures naturally |
| **Computation** | Closed-form | O(N log N) via FFT |
| **OTM Accuracy** | Underprices | More accurate |

**Key Takeaway:** The volatility smile is not a market anomaly — it is a reflection of the fact that real asset returns are skewed and fat-tailed. The VG model captures this through two additional parameters (ν, θ), priced efficiently via the FFT.